# 05. annotation/prediction merge

이 notebook은 한 project 안에 여러 `annotation`과 `prediction`이 있을 때, 원하는 source들을 순서대로 읽어서 새 `prediction` 또는 새 `annotation`으로 합치는 용도입니다.

안전 기본값은 `DRY_RUN=True`입니다. 이 상태에서는 어떤 task에 몇 개 bbox가 쓰일지 계산만 하고 Label Studio에 POST하지 않습니다.

핵심 용어:
- `SrcAnn(user_id, min_lead)`: 특정 사용자가 만든 annotation 중 lead_time 조건을 만족하는 최신 annotation을 source로 사용합니다.
- `SrcPred(model_ver, score_thr)`: 특정 model_version prediction 중 score threshold 이상 bbox만 source로 사용합니다.
- `sources`: source list 순서입니다. `resolve="keep_earlier"`에서는 앞에 있는 source가 우선권을 가집니다.
- `resolve="higher_score"`: 겹치는 bbox가 있을 때 score가 높은 bbox를 남깁니다.
- `CLASS_GROUPS`: 서로 다른 class라도 같은 그룹이면 겹치는 bbox 충돌 대상으로 봅니다.
- `GROUPED_IOU_THR`: 같은 group 안에서 겹침을 판단할 IoU threshold입니다.


In [ ]:
from pathlib import Path
import sys

# notebook을 repo 어느 위치에서 열어도 src package를 찾기 위한 bootstrap입니다.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from labelstudio_bbox_tools.config import settings_from_env

settings = settings_from_env(REPO_ROOT / ".env")
print("repo:", REPO_ROOT)
print("Label Studio URL:", settings.url)
print("doc_root:", settings.doc_root)


In [ ]:
from labelstudio_bbox_tools.merge.ann_pred import SrcAnn, SrcPred, merge_to_new_prediction, merge_to_new_annotation

PROJECT_ID = 0

# 예시 1: prediction 두 개를 합쳐 새 prediction으로 저장
src_p1 = SrcPred(model_ver="model_version_a", score_thr=0.50)
src_p2 = SrcPred(model_ver="model_version_b", score_thr=0.50)

# 예시 2: annotation을 source로 쓸 때
# src_ann = SrcAnn(user_id=1, min_lead=0.0)

sources = [src_p1, src_p2]

IOU_BASE = 0.50
IOU_PER_CLASS = {}

CLASS_GROUPS = [
    ["worker", "worker_no_vest", "worker_no_helmet_no_vest", "signalman", "signalman_no_red"],
    ["small_worker", "small_worker_no_vest", "small_worker_no_helmet_no_vest", "small_signalman", "small_signalman_no_red"],
]
GROUPED_IOU_THR = 0.40

RESOLVE = "keep_earlier"  # "keep_earlier" 또는 "higher_score"
NEW_MODEL_VER = "merged_prediction_name"
META_TAG = NEW_MODEL_VER

MAX_TASKS = 20
DRY_RUN = True


In [ ]:
summary = merge_to_new_prediction(
    ls_url=settings.url,
    api_key=settings.api_key,
    project_id=PROJECT_ID,
    sources=sources,
    iou_thr_base=IOU_BASE,
    iou_thr_per_class=IOU_PER_CLASS,
    class_groups=CLASS_GROUPS,
    grouped_iou_thr=GROUPED_IOU_THR,
    resolve=RESOLVE,
    new_model_ver=NEW_MODEL_VER,
    meta_tag=META_TAG,
    doc_root=settings.doc_root,
    dry_run=DRY_RUN,
    max_tasks=MAX_TASKS,
)
summary.as_dict()


In [ ]:
# 새 annotation으로 저장하고 싶을 때만 사용합니다.
# annotation은 completed_by 값이 필요하므로, 실사용 전 프로젝트 사용자 id 정책을 확인하세요.
RUN_ANNOTATION_MERGE = False

if RUN_ANNOTATION_MERGE:
    summary = merge_to_new_annotation(
        ls_url=settings.url,
        api_key=settings.api_key,
        project_id=PROJECT_ID,
        sources=sources,
        iou_thr_base=IOU_BASE,
        iou_thr_per_class=IOU_PER_CLASS,
        class_groups=CLASS_GROUPS,
        grouped_iou_thr=GROUPED_IOU_THR,
        resolve=RESOLVE,
        new_import_id="merged_annotation_import_id",
        meta_tag="merged_annotation_tag",
        doc_root=settings.doc_root,
        dry_run=True,
        max_tasks=MAX_TASKS,
    )
    display(summary.as_dict())
